# Analise e tratamento dados bronze b_contas_pagar.csv

In [9]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [10]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
BRONZE_DIR =  os.path.join(DATA_DIR, 'bronze')

In [11]:
df = pd.read_csv(os.path.join(BRONZE_DIR, 'b_contas_pagar.csv'))
df.shape

(404, 10)

## Analise exploratória

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_pagar    404 non-null    str    
 1   id_fornecedor      404 non-null    str    
 2   data_emissao       404 non-null    str    
 3   data_vencimento    404 non-null    str    
 4   data_pagamento     404 non-null    str    
 5   categoria_despesa  404 non-null    str    
 6   valor_titulo       404 non-null    float64
 7   valor_pago         404 non-null    float64
 8   status             404 non-null    str    
 9   forma_pagamento    404 non-null    str    
dtypes: float64(2), str(8)
memory usage: 31.7 KB


In [14]:
df.head()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000001,F0005,2026-05-01,2026-05-22,2026-05-25,Mercadorias,12165.76,12165.76,Pago,Boleto
1,CP000002,F0002,2026-03-02,2026-04-01,1900-01-01,Tecnologia,16428.88,0.00,Em aberto,PIX
2,CP000003,F0014,2026-03-30,2026-04-13,2026-04-12,Aluguel,19536.09,19536.09,Pago,Boleto
3,CP000004,F0022,2026-05-02,2026-06-16,1900-01-01,Marketing,13913.86,0.00,Em aberto,Transferência
4,CP000005,F0019,2026-05-25,2026-06-15,1900-01-01,Aluguel,12418.04,0.00,Em aberto,Boleto


In [13]:
df.isnull().sum()

id_titulo_pagar      0
id_fornecedor        0
data_emissao         0
data_vencimento      0
data_pagamento       0
categoria_despesa    0
valor_titulo         0
valor_pago           0
status               0
forma_pagamento      0
dtype: int64

## Tratamento de dados

In [15]:
df_original = df.copy()

In [19]:
# campos datas

df['data_emissao'] = pd.to_datetime(df['data_emissao'], format='mixed')
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], format='mixed')
df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], format='mixed')


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_titulo_pagar    404 non-null    str           
 1   id_fornecedor      404 non-null    str           
 2   data_emissao       404 non-null    datetime64[us]
 3   data_vencimento    404 non-null    datetime64[us]
 4   data_pagamento     404 non-null    datetime64[us]
 5   categoria_despesa  404 non-null    str           
 6   valor_titulo       404 non-null    float64       
 7   valor_pago         404 non-null    float64       
 8   status             404 non-null    str           
 9   forma_pagamento    404 non-null    str           
dtypes: datetime64[us](3), float64(2), str(5)
memory usage: 31.7 KB


### Criando campos extras

In [21]:
# qtde_dias_emissao

df['qtde_dias_emissao'] = df['data_vencimento'] - df['data_emissao']


In [22]:
# qtde_dias_pagamento_atraso

df['qtde_dias_pagamento_atraso'] = df['data_pagamento'] - df['data_vencimento']


In [23]:
df.head()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento,qtde_dias_emissao,qtde_dias_pagamento_atraso
0,CP000001,F0005,2026-05-01,2026-05-22,2026-05-25,Mercadorias,12165.76,12165.76,Pago,Boleto,21 days,3 days
1,CP000002,F0002,2026-03-02,2026-04-01,1900-01-01,Tecnologia,16428.88,0.00,Em aberto,PIX,30 days,-46111 days
2,CP000003,F0014,2026-03-30,2026-04-13,2026-04-12,Aluguel,19536.09,19536.09,Pago,Boleto,14 days,-1 days
3,CP000004,F0022,2026-05-02,2026-06-16,1900-01-01,Marketing,13913.86,0.00,Em aberto,Transferência,45 days,-46187 days
4,CP000005,F0019,2026-05-25,2026-06-15,1900-01-01,Aluguel,12418.04,0.00,Em aberto,Boleto,21 days,-46186 days
